In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score, classification_report
import tensorflow as tf
import typing
from keras._tf_keras import keras
from keras import layers, Model, callbacks
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

In [2]:

data = pd.read_csv('C:/Users/Asus/projects/lstm-learner-progression/data/raw/learner_data.csv')
print(data.head())

   learner_id  timestamp                  module  quiz_score  engagement_rate  \
0           0          4           deep_learning       42.06            0.550   
1           0          8              npm_basics       59.24            0.554   
2           1          4              npm_basics       32.17            0.348   
3           1          8  reinforcement_learning       34.48            0.182   
4           2          4         data_structures       44.98            0.508   

   hint_count  session_duration  correct_attempts  incorrect_attempts  \
0           5              71.5                12                   2   
1           1              63.9                12                   4   
2           4              38.6                 2                   3   
3           4              53.7                10                   8   
4           1              39.6                 3                   5   

   ability_latent  
0           0.616  
1           0.616  
2           0.

In [3]:
class BahdanauAttention(layers.Layer):

    def __init__(self, units: int=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias =False)
        self.V = layers.Dense(1, use_bias=False)

    def call(self, encoder_outputs):
        score = self.V(tf.nn.tanh(self.W(encoder_outputs)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights*encoder_outputs, axis=1)
        return context, tf.squeeze(weights, axis=1)

In [4]:
def build_lstm_model(seq_len: int=10,
                     num_features: int=6,
                     num_modules: int=6,
                     embed_dim: int=8,
                     lstm_units_1: int=128,
                     lstm_units_2: int=64,
                     attention_units: int=64,
                     dropout_rate: float=0.3,
                     l2_reg: float=1e-4) -> Model:

    num_input = keras.Input(shape=(seq_len, num_features), name="numerical_input")
    cat_input = keras.Input(shape=(seq_len,), name="module_id_input")

    embedding = layers.Embedding(
        input_dim = num_modules +1,
        output_dim = embed_dim, 
        name = "module_embedding"
    )(cat_input)

    x = layers.Concatenate(name="feature_fusion")([num_input, embedding])
    x = layers.LSTM(units=lstm_units_1,return_sequences=True,dropout=dropout_rate,recurrent_dropout=0.1,kernel_regularizer=reg,name="lstm_layer_1")(x)
    x=layers.LayerNormalization(name="layer_norm_1")(x)
    x=layers.LSTM(units=lstm_units_2,return_sequences=True, dropout=dropout_rate,recurrent_dropout=0.1,kernel_regularizer=reg,name="lstm_layer_2")(x)
    x=layers.LayerNormalization(name="layer_norm_2")(x)

    attention = BahdanauAttention(units=attention_units, name="attention")
    context, attn_weights = attention(x)
    
    shared=layers.Dense(64, activation="relu", kernel_regularizer=reg, name="shared_dense")(context)
    shared=layers.Dropout(dropout_rate,name="shared_dropout")(shared)

    perf_hidden = layers.Dense(32, activation="relu", name="perf_hidden")(shared)
    performance_output=layers.Dense(1, activation="linear",name="performance_output")(perf_hidden)

    mastery_hidden=layers.Dense(32,activation="relu", name="mastery_hidden")(shared)
    mastery_output=layers.Dense(1, activation="sigmoid", name="mastery_output")(mastery_hidden)

    dropout_hidden=layers.Dense(32, activation="relu", name="dropout_hidden")(shared)
    dropout_output=layers.Dense(1, activation="sigmoid", name="dropout_output")(dropout_hidden)

    model = Model(input=[num_input, cat_input], 
                  outputs={"performance_score": performance_output,
                           "mastery_prob": mastery_output,
                           "dropout_risk": dropout_output,}, name="LearnerProgressionLSTM")
    return model

In [5]:
# TRAINING
def compile_model(model: Model, learning_rate: float = 1e-3) -> Model:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss={
            "performance_score": keras.losses.MeanSquaredError(),
            "mastery_prob":      keras.losses.BinaryCrossentropy(),
            "dropout_risk":      keras.losses.BinaryCrossentropy(),
        },
        loss_weights={
            "performance_score": 0.3,
            "mastery_prob":      0.35,
            "dropout_risk":      0.35,
        },
        metrics={
            "performance_score": [keras.metrics.RootMeanSquaredError(name="rmse")],
            "mastery_prob":      [keras.metrics.AUC(name="auc")],
            "dropout_risk":      [keras.metrics.AUC(name="auc")],
        }
    )
    return model


def get_callbacks(checkpoint_path: str = "artifacts/checkpoints/best_model.keras") -> list:
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=0
        ),
    ]

In [6]:
def evaluate_model(model: Model, X_num, X_cat, y_true: np.ndarray, split_name: str = "Test"):
    """
    Print a formatted evaluation report for all three output heads.
    """
    preds = model.predict([X_num, X_cat], verbose=0)

    perf_pred    = preds["performance_score"].squeeze()
    mastery_pred = preds["mastery_prob"].squeeze()
    dropout_pred = preds["dropout_risk"].squeeze()

    perf_true    = y_true[:, 0]
    mastery_true = y_true[:, 1].astype(int)
    dropout_true = y_true[:, 2].astype(int)

    rmse = np.sqrt(mean_squared_error(perf_true, perf_pred))
    try:
        mastery_auc = roc_auc_score(mastery_true, mastery_pred)
        dropout_auc = roc_auc_score(dropout_true, dropout_pred)
    except ValueError:
        mastery_auc = dropout_auc = float("nan")

    mastery_bin = (mastery_pred >= 0.5).astype(int)
    dropout_bin = (dropout_pred >= 0.5).astype(int)

    sep = "─" * 54
    print(f"\n{'═'*54}")
    print(f"  {split_name} Evaluation Results")
    print(f"{'═'*54}")
    print(f"\n  Performance Score (Regression)")
    print(f"  {sep}")
    print(f"  RMSE : {rmse:>8.3f}  (scale 0-100)")
    print(f"  MAE  : {np.mean(np.abs(perf_true - perf_pred)):>8.3f}")

    print(f"\n  Mastery Probability (Classification)")
    print(f"  {sep}")
    print(f"  AUC  : {mastery_auc:>8.4f}")
    print(f"\n{classification_report(mastery_true, mastery_bin, target_names=['Not Mastered','Mastered'], indent=2)}")

    print(f"\n  Dropout Risk (Classification)")
    print(f"  {sep}")
    print(f"  AUC  : {dropout_auc:>8.4f}")
    print(f"\n{classification_report(dropout_true, dropout_bin, target_names=['Retained','At Risk'], indent=2)}")
    print(f"{'═'*54}\n")

    return {
        "rmse_performance": rmse,
        "auc_mastery":      mastery_auc,
        "auc_dropout":      dropout_auc,
    }


In [7]:
def get_attention_model(trained_model: Model) -> Model:
    """
    Returns a sub-model that also outputs attention weights.
    Useful for explainability: inspect which session (timestep)
    the model focused on for each prediction.
    """
    attn_output = trained_model.get_layer("attention").output  # (context, weights)
    return Model(
        inputs=trained_model.inputs,
        outputs={
            "performance_score": trained_model.output["performance_score"],
            "mastery_prob":      trained_model.output["mastery_prob"],
            "dropout_risk":      trained_model.output["dropout_risk"],
            "attention_weights": attn_output[1],  # (batch, T)
        },
        name="LearnerProgressionLSTM_Explainable"
    )

In [ ]:
class LearnerPredictor:
    """
    Thin wrapper around the trained model for single-learner
    real-time inference (as called by src/inference/predictor.py).
    """

    RISK_THRESHOLDS = {
        "high":   0.65,
        "medium": 0.40,
        "low":    0.0,
    }

    def __init__(self, model: Model, scaler: MinMaxScaler):
        self.model  = model
        self.scaler = scaler

    def predict(self, num_seq: np.ndarray, cat_seq: np.ndarray) -> dict:
        """
        Parameters
        ----------
        num_seq : (seq_len, 6)  raw (unscaled) numerical features
        cat_seq : (seq_len,)    module_id integers

        Returns
        -------
        dict with performance_score, mastery_prob, dropout_risk,
        dropout_tier, and a recommended intervention string.
        """
        scaled = self.scaler.transform(num_seq)
        X_num  = scaled[np.newaxis]             # (1, T, 6)
        X_cat  = cat_seq[np.newaxis]            # (1, T)

        preds = self.model.predict([X_num, X_cat], verbose=0)
        perf    = float(preds["performance_score"][0, 0])
        mastery = float(preds["mastery_prob"][0, 0])
        dropout = float(preds["dropout_risk"][0, 0])

        tier = "low"
        for t, thresh in self.RISK_THRESHOLDS.items():
            if dropout >= thresh:
                tier = t
                break

        intervention = self._recommend_intervention(mastery, dropout, perf)

        return {
            "performance_score": round(perf, 2),
            "mastery_prob":      round(mastery, 4),
            "dropout_risk":      round(dropout, 4),
            "dropout_tier":      tier,
            "intervention":      intervention,
        }

    @staticmethod
    def _recommend_intervention(mastery, dropout, perf) -> str:
        if dropout > 0.65:
            return "ALERT_INSTRUCTOR"
        if mastery < 0.4 and perf < 60:
            return "ASSIGN_REMEDIAL_MODULE"
        if dropout > 0.4:
            return "SEND_MOTIVATIONAL_NUDGE"
        if mastery > 0.75:
            return "AWARD_BADGE_AND_ADVANCE"
        return "CONTINUE_STANDARD_PATH"